# 🎧 AuraNet Training Notebook

**Auditory Universal Reflex and Analysis Network**

A lightweight, strictly causal CRN for real-time audio enhancement with biomimetic design principles.

---

## Features
- Train on custom audio datasets from Google Drive
- Automatic GPU detection and utilization
- Checkpoint saving and resume support
- Inference and audio playback
- Debug mode for quick testing

---

## 📦 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchaudio librosa numpy soundfile tqdm pyyaml

# Verify installation
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ Torchaudio: {torchaudio.__version__}")
print(f"✅ Librosa: {librosa.__version__}")

## 🔧 2. Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these settings as needed
# =============================================================================

CONFIG = {
    # Debug mode - set to True for quick testing with small dataset
    "debug_mode": False,

    # Audio settings
    "audio": {
        "sample_rate": 16000,
    },

    # STFT settings (strictly causal)
    "stft": {
        "window_size": 160,   # 10ms at 16kHz
        "hop_size": 80,       # 5ms at 16kHz
        "n_fft": 256,
    },

    # Model architecture
    "model": {
        "encoder": {
            "in_channels": 2,
            "channels": [16, 32, 64, 128],
        },
        "bottleneck": {
            "hidden_size": 256,
            "num_layers": 1,
        },
        "decoder": {
            "channels": [64, 32, 16, 2],
        },
        "wdrc": {
            "hidden_dims": [128, 64],
        },
    },

    # Loss function weights
    "loss": {
        "complex_mse": 1.0,
        "multi_resolution_stft": 0.5,
        "loudness_envelope": 0.3,
        "temporal_coherence": 0.2,
    },

    # Training settings
    "training": {
        "batch_size": 16,
        "num_epochs": 25,  # Recommended Colab default; use 20 for a faster run
        "learning_rate": 0.001,
        "weight_decay": 0.01,
        "gradient_clip": 5.0,
        "use_amp": True,
    },

    # Data settings
    "data": {
        "segment_length": 3.0,   # seconds
        "snr_range": [-5, 20],   # dB
        "num_workers": 2,
    },

    # Paths (will be set after mounting Drive)
    "paths": {
        "dataset_root": "/content/drive/MyDrive/auranet_dataset",
        "checkpoint_dir": "/content/drive/MyDrive/auranet_checkpoints",
    },
}

# Debug mode settings - reduced for quick testing
if CONFIG["debug_mode"]:
    CONFIG["training"]["batch_size"] = 4
    CONFIG["training"]["num_epochs"] = 2

## ⚡ 3. GPU Detection

In [ ]:
# =============================================================================
# GPU DETECTION AND SETUP
# =============================================================================

def setup_device():
    """Detect and setup the best available device."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"🎮 GPU Detected: {gpu_name}")
        print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
        print(f"🔥 CUDA Version: {torch.version.cuda}")
    else:
        device = torch.device("cpu")
        print("⚠️ No GPU detected, using CPU (training will be slow)")
        print("💡 Tip: Go to Runtime > Change runtime type > GPU")

    return device

DEVICE = setup_device()
print(f"\n✅ Using device: {DEVICE}")

## 📁 4. Google Drive Integration

In [ ]:
# =============================================================================
# MOUNT GOOGLE DRIVE
# =============================================================================

from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create directories if they don't exist
os.makedirs(CONFIG["paths"]["checkpoint_dir"], exist_ok=True)

print(f"\n📂 Dataset root: {CONFIG['paths']['dataset_root']}")
print(f"💾 Checkpoints: {CONFIG['paths']['checkpoint_dir']}")

# Check dataset structure
dataset_root = CONFIG["paths"]["dataset_root"]
expected_folders = ["speech", "music", "noise"]

print("\n📋 Checking dataset structure...")
for folder in expected_folders:
    path = os.path.join(dataset_root, folder)
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if f.endswith(('.wav', '.mp3', '.flac', '.ogg'))]
        print(f"  ✅ {folder}/: {len(files)} audio files")
    else:
        print(f"  ⚠️ {folder}/: Not found - please create and add audio files")

## 🏗️ 5. Model Architecture

In [ ]:
# =============================================================================
# AURANET MODEL DEFINITION
# =============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional, Dict, Any
import math


class CausalSTFT(nn.Module):
    """Strictly causal STFT for real-time processing."""

    def __init__(self, n_fft=256, hop_length=80, win_length=160, window="hann"):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.n_freqs = n_fft // 2 + 1

        if window == "hann":
            win = torch.hann_window(n_fft, periodic=True)
        else:
            win = torch.hann_window(n_fft, periodic=True)
        self.register_buffer("window", win)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        if waveform.dim() == 3:
            waveform = waveform.squeeze(1)

        pad_length = self.n_fft - 1
        waveform_padded = F.pad(waveform, (pad_length, 0), mode="constant", value=0)

        stft_complex = torch.stft(
            waveform_padded,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.window,
            center=False,
            return_complex=True,
            onesided=True,
        )

        stft_real = stft_complex.real
        stft_imag = stft_complex.imag
        stft_out = torch.stack([stft_real, stft_imag], dim=1)
        stft_out = stft_out.permute(0, 1, 3, 2)

        return stft_out

    def inverse(self, stft_tensor: torch.Tensor, length: Optional[int] = None) -> torch.Tensor:
        batch_size, _, n_frames, n_freqs = stft_tensor.shape
        stft_tensor = stft_tensor.permute(0, 1, 3, 2)
        stft_real = stft_tensor[:, 0, :, :]
        stft_imag = stft_tensor[:, 1, :, :]
        stft_complex = torch.complex(stft_real, stft_imag)

        waveform = torch.istft(
            stft_complex,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.n_fft,
            window=self.window,
            center=True,
            onesided=True,
            length=length,
            return_complex=False,
        )
        return waveform


class CausalConv2d(nn.Module):
    """2D Convolution with causal padding in time dimension."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=(1, 1),
                 dilation=(1, 1), groups=1, bias=True):
        super().__init__()
        self.pad_time = (kernel_size[0] - 1) * dilation[0]
        self.pad_freq = ((kernel_size[1] - 1) * dilation[1]) // 2
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size,
                              stride=stride, padding=(0, self.pad_freq),
                              dilation=dilation, groups=groups, bias=bias)

    def forward(self, x):
        x = F.pad(x, (0, 0, self.pad_time, 0))
        return self.conv(x)


class CausalTransposedConv2d(nn.Module):
    """Transposed convolution with causal handling."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=(1, 2),
                 groups=1, bias=True):
        super().__init__()
        output_padding = (0, stride[1] - 1) if stride[1] > 1 else (0, 0)
        padding = ((kernel_size[0] - 1), kernel_size[1] // 2)
        self.conv_t = nn.ConvTranspose2d(in_channels, out_channels, kernel_size,
                                         stride=stride, padding=padding,
                                         output_padding=output_padding,
                                         groups=groups, bias=bias)

    def forward(self, x):
        return self.conv_t(x)


class DepthwiseSeparableConv2d(nn.Module):
    """Depthwise separable convolution with causal padding."""

    def __init__(self, in_channels, out_channels, kernel_size=(3, 3),
                 stride=(1, 1), bias=False):
        super().__init__()
        self.depthwise = CausalConv2d(in_channels, in_channels, kernel_size,
                                       stride=stride, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, (1, 1), bias=bias)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class DepthwiseSeparableTransposedConv2d(nn.Module):
    """Depthwise separable transposed convolution."""

    def __init__(self, in_channels, out_channels, kernel_size=(3, 3),
                 stride=(1, 2), bias=False):
        super().__init__()
        self.pointwise = nn.Conv2d(in_channels, out_channels, (1, 1), bias=False)
        self.depthwise_t = CausalTransposedConv2d(out_channels, out_channels, kernel_size,
                                                  stride=stride, groups=out_channels, bias=bias)

    def forward(self, x):
        return self.depthwise_t(self.pointwise(x))


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3, 3), stride=(1, 2)):
        super().__init__()
        self.conv = DepthwiseSeparableConv2d(in_channels, out_channels, kernel_size, stride)
        self.norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.PReLU(out_channels)

    def forward(self, x):
        return self.activation(self.norm(self.conv(x)))


class Encoder(nn.Module):
    def __init__(self, in_channels=2, channels=(16, 32, 64, 128), kernel_size=(3, 3)):
        super().__init__()
        blocks = []
        current = in_channels
        for ch in channels:
            blocks.append(EncoderBlock(current, ch, kernel_size, stride=(1, 2)))
            current = ch
        self.blocks = nn.ModuleList(blocks)
        self.out_channels = channels[-1]

    def forward(self, x):
        skips = []
        for block in self.blocks:
            x = block(x)
            skips.append(x)
        return x, skips


class TemporalBottleneck(nn.Module):
    def __init__(self, input_channels=128, input_freq_bins=9, hidden_size=256,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.input_size = input_channels * input_freq_bins
        self.hidden_size = hidden_size
        self.gru = nn.GRU(self.input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0, bidirectional=False)
        self.projection = nn.Linear(hidden_size, self.input_size)

    def forward(self, x, hidden=None):
        B, C, T, F = x.shape
        x_flat = x.permute(0, 2, 1, 3).contiguous().view(B, T, -1)
        gru_out, hidden_out = self.gru(x_flat, hidden)
        projected = self.projection(gru_out).view(B, T, C, F)
        return projected.permute(0, 2, 1, 3).contiguous(), gru_out, hidden_out


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels=None,
                 kernel_size=(3, 3), stride=(1, 2), use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        actual_in = in_channels + (skip_channels or in_channels) if use_skip else in_channels
        self.conv = DepthwiseSeparableTransposedConv2d(actual_in, out_channels, kernel_size, stride)
        self.norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.PReLU(out_channels)

    def forward(self, x, skip=None):
        if self.use_skip and skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.activation(self.norm(self.conv(x)))


class Decoder(nn.Module):
    def __init__(self, in_channels=128, channels=(64, 32, 16, 2),
                 encoder_channels=(16, 32, 64, 128), kernel_size=(3, 3), use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        skip_channels = list(reversed(encoder_channels))

        blocks = []
        current = in_channels
        for i, ch in enumerate(channels[:-1]):
            skip_ch = skip_channels[i] if use_skip and i < len(skip_channels) else None
            blocks.append(DecoderBlock(current, ch, skip_ch, kernel_size, (1, 2), use_skip))
            current = ch
        self.blocks = nn.ModuleList(blocks)

        final_skip_ch = skip_channels[len(channels) - 1] if use_skip and len(channels) - 1 < len(skip_channels) else current
        final_in = current + final_skip_ch if use_skip else current
        self.output_conv = nn.Sequential(
            DepthwiseSeparableTransposedConv2d(final_in, channels[-1], (3, 3), (1, 2)),
            nn.Tanh(),
        )

    def forward(self, x, skip_connections):
        skips = skip_connections[::-1]
        for i, block in enumerate(self.blocks):
            x = block(x, skips[i] if i < len(skips) else None)

        if self.use_skip and len(skips) > len(self.blocks):
            last_skip = skips[len(self.blocks)]
            if x.shape[2:] != last_skip.shape[2:]:
                x = F.interpolate(x, size=last_skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, last_skip], dim=1)
        elif self.use_skip:
            x = torch.cat([x, x], dim=1)

        return self.output_conv(x)


class NeuralWDRC(nn.Module):
    """Neural Wide Dynamic Range Compression sidechain."""

    def __init__(self, input_dim=256, hidden_dims=(128, 64), output_dim=4):
        super().__init__()
        layers = []
        current = input_dim
        for dim in hidden_dims:
            layers.extend([nn.Linear(current, dim), nn.PReLU(), nn.Dropout(0.1)])
            current = dim
        layers.append(nn.Linear(current, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, gru_output):
        params = self.mlp(gru_output)
        return {
            "attack_coeff": torch.sigmoid(params[..., 0]),
            "release_coeff": torch.sigmoid(params[..., 1]),
            "compression_ratio": torch.clamp(F.softplus(params[..., 2]) + 1.0, 1.0, 20.0),
            "gain": torch.sigmoid(params[..., 3]) * 2.0,
        }


class AuraNet(nn.Module):
    """AuraNet: Auditory Universal Reflex and Analysis Network"""

    def __init__(self, in_channels=2, encoder_channels=(16, 32, 64, 128),
                 gru_hidden=256, gru_layers=1, decoder_channels=(64, 32, 16, 2),
                 wdrc_hidden=(128, 64), kernel_size=(3, 3), use_skip=True):
        super().__init__()

        self.encoder = Encoder(in_channels, encoder_channels, kernel_size)
        self.bottleneck = TemporalBottleneck(encoder_channels[-1], 9, gru_hidden, gru_layers)
        self.decoder = Decoder(encoder_channels[-1], decoder_channels, encoder_channels, kernel_size, use_skip)
        self.wdrc = NeuralWDRC(gru_hidden, wdrc_hidden)

    def forward(self, noisy_stft, hidden=None):
        B, C, T, F = noisy_stft.shape
        noisy_real, noisy_imag = noisy_stft[:, 0:1], noisy_stft[:, 1:2]

        encoded, skips = self.encoder(noisy_stft)
        bottleneck_out, gru_output, new_hidden = self.bottleneck(encoded, hidden)
        mask = self.decoder(bottleneck_out, skips)

        if mask.shape[-1] != F:
            mask = F.interpolate(mask, size=(T, F), mode='bilinear', align_corners=False)

        mask_real, mask_imag = mask[:, 0:1], mask[:, 1:2]
        enhanced_real = mask_real * noisy_real - mask_imag * noisy_imag
        enhanced_imag = mask_real * noisy_imag + mask_imag * noisy_real

        return torch.cat([enhanced_real, enhanced_imag], dim=1), self.wdrc(gru_output), new_hidden

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def create_auranet(config=None):
    """Factory function to create AuraNet."""
    if config is None:
        return AuraNet()

    model_cfg = config.get("model", {})
    return AuraNet(
        in_channels=model_cfg.get("encoder", {}).get("in_channels", 2),
        encoder_channels=tuple(model_cfg.get("encoder", {}).get("channels", [16, 32, 64, 128])),
        gru_hidden=model_cfg.get("bottleneck", {}).get("hidden_size", 256),
        gru_layers=model_cfg.get("bottleneck", {}).get("num_layers", 1),
        decoder_channels=tuple(model_cfg.get("decoder", {}).get("channels", [64, 32, 16, 2])),
        wdrc_hidden=tuple(model_cfg.get("wdrc", {}).get("hidden_dims", [128, 64])),
    )


# Create and verify model
model = create_auranet(CONFIG)
model = model.to(DEVICE)
print(f"\n✅ AuraNet created successfully!")
print(f"📊 Parameters: {model.count_parameters():,} (~{model.count_parameters()/1e6:.2f}M)")

## 📊 6. Loss Functions

In [ ]:
# =============================================================================
# LOSS FUNCTIONS
# =============================================================================

class ComplexMSELoss(nn.Module):
    """MSE on complex STFT (real + imaginary)."""

    def forward(self, pred, target):
        return F.mse_loss(pred[:, 0], target[:, 0]) + F.mse_loss(pred[:, 1], target[:, 1])


class STFTLoss(nn.Module):
    """Single-resolution STFT loss."""

    def __init__(self, fft_size=1024, hop_size=120, win_length=600):
        super().__init__()
        self.fft_size = fft_size
        self.hop_size = hop_size
        self.win_length = win_length
        self.register_buffer("window", torch.hann_window(win_length))

    def _stft_magnitude(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        window = self.window.to(x.device)
        spec = torch.stft(x, self.fft_size, self.hop_size, self.win_length,
                          window=window, center=True, return_complex=True)
        return torch.abs(spec)

    def forward(self, pred, target):
        min_len = min(pred.shape[-1], target.shape[-1])
        pred, target = pred[..., :min_len], target[..., :min_len]
        pred_mag, target_mag = self._stft_magnitude(pred), self._stft_magnitude(target)
        min_t = min(pred_mag.shape[-1], target_mag.shape[-1])
        pred_mag, target_mag = pred_mag[..., :min_t], target_mag[..., :min_t]

        sc_loss = torch.norm(target_mag - pred_mag, p='fro') / (torch.norm(target_mag, p='fro') + 1e-8)
        mag_loss = F.l1_loss(torch.log(pred_mag + 1e-8), torch.log(target_mag + 1e-8))
        return sc_loss + mag_loss


class MultiResolutionSTFTLoss(nn.Module):
    """Multi-resolution STFT loss."""

    def __init__(self, fft_sizes=[512, 1024, 2048], hop_sizes=[50, 120, 240],
                 win_lengths=[240, 600, 1200]):
        super().__init__()
        self.losses = nn.ModuleList([
            STFTLoss(fft, hop, win)
            for fft, hop, win in zip(fft_sizes, hop_sizes, win_lengths)
        ])

    def forward(self, pred, target):
        return sum(loss(pred, target) for loss in self.losses) / len(self.losses)


class LoudnessEnvelopeLoss(nn.Module):
    """Loss on loudness envelope."""

    def __init__(self, frame_size=160, hop_size=80):
        super().__init__()
        self.frame_size = frame_size
        self.hop_size = hop_size

    def _compute_envelope(self, x):
        if x.dim() == 3:
            x = x.squeeze(1)
        pad = self.frame_size - (x.shape[-1] % self.frame_size)
        if pad < self.frame_size:
            x = F.pad(x, (0, pad))
        frames = x.unfold(-1, self.frame_size, self.hop_size)
        return torch.sqrt(torch.mean(frames ** 2, dim=-1) + 1e-8)

    def forward(self, pred, target):
        min_len = min(pred.shape[-1], target.shape[-1])
        pred_env = self._compute_envelope(pred[..., :min_len])
        target_env = self._compute_envelope(target[..., :min_len])
        min_f = min(pred_env.shape[-1], target_env.shape[-1])
        return F.l1_loss(pred_env[..., :min_f], target_env[..., :min_f])


class TemporalCoherenceLoss(nn.Module):
    """Temporal coherence loss to prevent artifacts."""

    def forward(self, pred, target):
        min_len = min(pred.shape[-1], target.shape[-1])
        pred, target = pred[..., :min_len], target[..., :min_len]
        pred_diff = pred[..., 1:] - pred[..., :-1]
        target_diff = target[..., 1:] - target[..., :-1]
        return F.mse_loss(pred_diff, target_diff)


class AuraNetLoss(nn.Module):
    """Combined multi-task loss for AuraNet."""

    def __init__(self, weight_complex_mse=1.0, weight_stft=0.5,
                 weight_loudness=0.3, weight_temporal=0.2):
        super().__init__()
        self.weight_complex_mse = weight_complex_mse
        self.weight_stft = weight_stft
        self.weight_loudness = weight_loudness
        self.weight_temporal = weight_temporal

        self.complex_mse = ComplexMSELoss()
        self.multi_stft = MultiResolutionSTFTLoss()
        self.loudness = LoudnessEnvelopeLoss()
        self.temporal = TemporalCoherenceLoss()

    def forward(self, pred_stft, target_stft, pred_audio, target_audio):
        loss_complex = self.complex_mse(pred_stft, target_stft) * self.weight_complex_mse
        loss_stft = self.multi_stft(pred_audio, target_audio) * self.weight_stft
        loss_loud = self.loudness(pred_audio, target_audio) * self.weight_loudness
        loss_temp = self.temporal(pred_audio, target_audio) * self.weight_temporal

        total = loss_complex + loss_stft + loss_loud + loss_temp
        return total, {
            "total": total,
            "complex_mse": loss_complex,
            "multi_res_stft": loss_stft,
            "loudness_envelope": loss_loud,
            "temporal_coherence": loss_temp,
        }


print("✅ Loss functions defined")

## 📂 7. Dataset

In [ ]:
# =============================================================================
# DATASET AND DATA LOADING
# =============================================================================

import glob
import random
from torch.utils.data import Dataset, DataLoader


def load_audio(path, sample_rate=16000):
    """Load and resample audio file."""
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)
    return waveform, sample_rate


def mix_audio_with_noise(clean, noise, snr_db):
    """Mix clean audio with noise at specified SNR."""
    if clean.dim() == 1:
        clean = clean.unsqueeze(0)
    if noise.dim() == 1:
        noise = noise.unsqueeze(0)

    clean_len = clean.shape[-1]
    if noise.shape[-1] < clean_len:
        repeats = (clean_len // noise.shape[-1]) + 1
        noise = noise.repeat(1, repeats)

    if noise.shape[-1] > clean_len:
        start = random.randint(0, noise.shape[-1] - clean_len)
        noise = noise[:, start:start + clean_len]

    clean_rms = torch.sqrt(torch.mean(clean ** 2) + 1e-8)
    noise_rms = torch.sqrt(torch.mean(noise ** 2) + 1e-8)
    target_noise_rms = clean_rms / (10 ** (snr_db / 20))
    scaled_noise = noise * (target_noise_rms / (noise_rms + 1e-8))

    return clean + scaled_noise, scaled_noise


def generate_synthetic_noise(length, noise_type="white"):
    """Generate synthetic noise."""
    if noise_type == "white":
        return torch.randn(1, length)
    elif noise_type == "pink":
        white = torch.randn(length)
        fft = torch.fft.rfft(white)
        freqs = torch.fft.rfftfreq(length)
        fft = fft / (freqs + 1e-8).sqrt()
        fft[0] = 0
        return torch.fft.irfft(fft, n=length).unsqueeze(0)
    else:  # brown
        white = torch.randn(length)
        return torch.cumsum(white, dim=0).unsqueeze(0) / 100


class AuraNetDataset(Dataset):
    """Dataset for AuraNet training with support for real and synthetic data."""

    def __init__(self, clean_dirs, noise_dir=None, sample_rate=16000,
                 segment_length=3.0, snr_range=(-5.0, 20.0), n_fft=256,
                 hop_length=80, synthetic_mode=False, num_synthetic=1000):
        super().__init__()

        self.sample_rate = sample_rate
        self.segment_samples = int(segment_length * sample_rate)
        self.snr_range = snr_range
        self.synthetic_mode = synthetic_mode
        self.stft = CausalSTFT(n_fft, hop_length, n_fft // 2 + 1)

        if synthetic_mode:
            self.clean_files = list(range(num_synthetic))
            self.noise_files = list(range(num_synthetic))
        else:
            self.clean_files = []
            for d in (clean_dirs if isinstance(clean_dirs, list) else [clean_dirs]):
                if os.path.exists(d):
                    for ext in ['*.wav', '*.mp3', '*.flac', '*.ogg']:
                        self.clean_files.extend(glob.glob(os.path.join(d, '**', ext), recursive=True))

            self.noise_files = []
            if noise_dir and os.path.exists(noise_dir):
                for ext in ['*.wav', '*.mp3', '*.flac', '*.ogg']:
                    self.noise_files.extend(glob.glob(os.path.join(noise_dir, '**', ext), recursive=True))

            if len(self.clean_files) == 0:
                print("⚠️ No clean files found. Switching to synthetic mode.")
                self.synthetic_mode = True
                self.clean_files = list(range(num_synthetic))
                self.noise_files = list(range(num_synthetic))

    def _generate_synthetic_clean(self):
        length = self.segment_samples
        t = torch.linspace(0, self.segment_samples / self.sample_rate, length)
        audio = torch.zeros(1, length)

        num_harm = random.randint(3, 8)
        fund = random.uniform(100, 400)
        for h in range(1, num_harm + 1):
            audio += (1.0 / h) * torch.sin(2 * 3.14159 * fund * h * t + random.uniform(0, 6.28)).unsqueeze(0)

        mod_freq = random.uniform(2, 10)
        audio *= (0.5 + 0.5 * torch.sin(2 * 3.14159 * mod_freq * t)).unsqueeze(0)
        return audio / (audio.abs().max() + 1e-8) * 0.8

    def _generate_synthetic_noise(self):
        noise_type = random.choice(["white", "pink", "brown"])
        return generate_synthetic_noise(self.segment_samples, noise_type)

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        if self.synthetic_mode:
            clean_audio = self._generate_synthetic_clean()
            noise_audio = self._generate_synthetic_noise()
        else:
            clean_audio, _ = load_audio(self.clean_files[idx], self.sample_rate)
            noise_idx = random.randint(0, max(0, len(self.noise_files) - 1))
            noise_audio = (
                load_audio(self.noise_files[noise_idx], self.sample_rate)[0]
                if self.noise_files
                else self._generate_synthetic_noise()
            )

        # Random crop
        if clean_audio.shape[-1] >= self.segment_samples:
            start = random.randint(0, clean_audio.shape[-1] - self.segment_samples)
            clean_audio = clean_audio[:, start:start + self.segment_samples]
        else:
            clean_audio = F.pad(clean_audio, (0, self.segment_samples - clean_audio.shape[-1]))

        if noise_audio.shape[-1] >= self.segment_samples:
            start = random.randint(0, noise_audio.shape[-1] - self.segment_samples)
            noise_audio = noise_audio[:, start:start + self.segment_samples]
        else:
            noise_audio = F.pad(noise_audio, (0, self.segment_samples - noise_audio.shape[-1]))

        snr = random.uniform(self.snr_range[0], self.snr_range[1])
        noisy_audio, _ = mix_audio_with_noise(clean_audio, noise_audio, snr)

        noisy_stft = self.stft(noisy_audio).squeeze(0)
        clean_stft = self.stft(clean_audio).squeeze(0)

        return {
            "noisy_stft": noisy_stft,
            "clean_stft": clean_stft,
            "noisy_audio": noisy_audio.squeeze(0),
            "clean_audio": clean_audio.squeeze(0),
            "snr": torch.tensor(snr),
        }


# Create datasets
dataset_root = CONFIG["paths"]["dataset_root"]
clean_dirs = [
    os.path.join(dataset_root, "speech"),
    os.path.join(dataset_root, "music"),
]
noise_dir = os.path.join(dataset_root, "noise")

# Check if we should use synthetic mode
use_synthetic = CONFIG["debug_mode"] or not os.path.exists(dataset_root)

train_dataset = AuraNetDataset(
    clean_dirs=clean_dirs,
    noise_dir=noise_dir,
    sample_rate=CONFIG["audio"]["sample_rate"],
    segment_length=CONFIG["data"]["segment_length"],
    snr_range=tuple(CONFIG["data"]["snr_range"]),
    n_fft=CONFIG["stft"]["n_fft"],
    hop_length=CONFIG["stft"]["hop_size"],
    synthetic_mode=use_synthetic,
    num_synthetic=100 if CONFIG["debug_mode"] else 1000,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["training"]["batch_size"],
    shuffle=True,
    num_workers=CONFIG["data"]["num_workers"],
    pin_memory=True,
)

print(f"\n✅ Dataset created")
print(f"📊 Total samples: {len(train_dataset)}")
print(f"📦 Batches per epoch: {len(train_loader)}")
print(f"🎲 Mode: {'Synthetic' if use_synthetic else 'Real data'}")

## 🏋️ 8. Training Pipeline

In [ ]:
# =============================================================================
# TRAINING PIPELINE
# =============================================================================

from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
import time
from datetime import datetime


class AuraNetTrainer:
    """Trainer class for AuraNet."""

    def __init__(self, model, config, device):
        self.model = model
        self.config = config
        self.device = device

        # STFT for reconstruction
        stft_cfg = config.get("stft", {})
        self.stft = CausalSTFT(
            n_fft=stft_cfg.get("n_fft", 256),
            hop_length=stft_cfg.get("hop_size", 80),
        ).to(device)

        # Loss function
        loss_cfg = config.get("loss", {})
        self.criterion = AuraNetLoss(
            weight_complex_mse=loss_cfg.get("complex_mse", 1.0),
            weight_stft=loss_cfg.get("multi_resolution_stft", 0.5),
            weight_loudness=loss_cfg.get("loudness_envelope", 0.3),
            weight_temporal=loss_cfg.get("temporal_coherence", 0.2),
        )

        # Optimizer
        train_cfg = config.get("training", {})
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=train_cfg.get("learning_rate", 0.001),
            weight_decay=train_cfg.get("weight_decay", 0.01),
        )

        # Scheduler
        self.num_epochs = train_cfg.get("num_epochs", 100)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=self.num_epochs, eta_min=1e-6)

        # Mixed precision
        self.use_amp = train_cfg.get("use_amp", True) and device.type == "cuda"
        self.scaler = GradScaler() if self.use_amp else None

        # Gradient clipping
        self.grad_clip = train_cfg.get("gradient_clip", 5.0)

        # Checkpointing
        self.checkpoint_dir = config["paths"]["checkpoint_dir"]
        os.makedirs(self.checkpoint_dir, exist_ok=True)

        # State
        self.current_epoch = 0
        self.best_loss = float("inf")
        self.history = {"train_loss": [], "val_loss": []}

    def train_epoch(self, train_loader):
        """Train for one epoch."""
        self.model.train()
        epoch_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {self.current_epoch + 1}/{self.num_epochs}")
        for batch in pbar:
            noisy_stft = batch["noisy_stft"].to(self.device)
            clean_stft = batch["clean_stft"].to(self.device)
            noisy_audio = batch["noisy_audio"].to(self.device)
            clean_audio = batch["clean_audio"].to(self.device)

            self.optimizer.zero_grad()

            if self.use_amp:
                with autocast():
                    enhanced_stft, _, _ = self.model(noisy_stft)
                    enhanced_audio = self.stft.inverse(enhanced_stft)
                    loss, _ = self.criterion(enhanced_stft, clean_stft, enhanced_audio, clean_audio)

                self.scaler.scale(loss).backward()
                if self.grad_clip > 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                enhanced_stft, _, _ = self.model(noisy_stft)
                enhanced_audio = self.stft.inverse(enhanced_stft)
                loss, _ = self.criterion(enhanced_stft, clean_stft, enhanced_audio, clean_audio)

                loss.backward()
                if self.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                self.optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        return epoch_loss / len(train_loader)

    def save_checkpoint(self, filename, is_best=False):
        """Save model checkpoint."""
        checkpoint = {
            "epoch": self.current_epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "best_loss": self.best_loss,
            "config": self.config,
            "history": self.history,
        }
        if self.scaler:
            checkpoint["scaler_state_dict"] = self.scaler.state_dict()

        path = os.path.join(self.checkpoint_dir, filename)
        torch.save(checkpoint, path)
        print(f"💾 Saved checkpoint: {path}")

        if is_best:
            best_path = os.path.join(self.checkpoint_dir, "best_model.pt")
            torch.save(checkpoint, best_path)
            print(f"🏆 Saved best model: {best_path}")

    def load_checkpoint(self, path):
        """Load checkpoint and resume training."""
        checkpoint = torch.load(path, map_location=self.device)

        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        if self.scaler and "scaler_state_dict" in checkpoint:
            self.scaler.load_state_dict(checkpoint["scaler_state_dict"])

        self.current_epoch = checkpoint["epoch"]
        self.best_loss = checkpoint.get("best_loss", float("inf"))
        self.history = checkpoint.get("history", {"train_loss": [], "val_loss": []})

        print(f"✅ Resumed from epoch {self.current_epoch}")

    def train(self, train_loader, resume_from=None):
        """Full training loop."""
        # Resume if checkpoint provided
        if resume_from and os.path.exists(resume_from):
            self.load_checkpoint(resume_from)

        start_time = time.time()
        start_epoch = self.current_epoch

        print(f"\n🚀 Starting training from epoch {start_epoch + 1}")
        print(f"📊 Total epochs: {self.num_epochs}")
        print(f"⚡ Mixed precision: {self.use_amp}")
        print("-" * 50)

        for epoch in range(start_epoch, self.num_epochs):
            self.current_epoch = epoch

            # Train
            train_loss = self.train_epoch(train_loader)
            self.history["train_loss"].append(train_loss)

            # Update scheduler
            self.scheduler.step()

            # Check if best
            is_best = train_loss < self.best_loss
            if is_best:
                self.best_loss = train_loss

            # Log
            lr = self.optimizer.param_groups[0]['lr']
            print(f"\nEpoch {epoch + 1}/{self.num_epochs}")
            print(f"  Train Loss: {train_loss:.4f} | LR: {lr:.6f}")

            # Save checkpoint every 5 epochs and at end
            if (epoch + 1) % 5 == 0 or epoch == self.num_epochs - 1:
                self.save_checkpoint(f"checkpoint_epoch_{epoch + 1}.pt", is_best=is_best)

        elapsed = time.time() - start_time
        print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
        print(f"🏆 Best loss: {self.best_loss:.4f}")

        return self.history


# Create trainer
trainer = AuraNetTrainer(model, CONFIG, DEVICE)
print("\n✅ Trainer initialized")

## ▶️ 9. Run Training

In [ ]:
# =============================================================================
# RUN TRAINING
# =============================================================================

# Resume from checkpoint if exists
resume_path = os.path.join(CONFIG["paths"]["checkpoint_dir"], "best_model.pt")
if not os.path.exists(resume_path):
    resume_path = None

# Train!
history = trainer.train(train_loader, resume_from=resume_path)

## 📈 10. Plot Training Results

In [ ]:
# =============================================================================
# PLOT TRAINING HISTORY
# =============================================================================

import matplotlib.pyplot as plt

if history["train_loss"]:
    plt.figure(figsize=(10, 5))
    plt.plot(history["train_loss"], label="Train Loss", marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("AuraNet Training Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\n📊 Final loss: {history['train_loss'][-1]:.4f}")
    print(f"📉 Best loss: {min(history['train_loss']):.4f}")
else:
    print("No training history to plot")

## 🎵 11. Inference & Audio Playback

In [ ]:
# =============================================================================
# INFERENCE
# =============================================================================

from google.colab import files
from IPython.display import Audio, display


class AuraNetInference:
    """Inference wrapper for AuraNet."""

    def __init__(self, model, config, device):
        self.model = model
        self.device = device
        self.sample_rate = config["audio"]["sample_rate"]

        stft_cfg = config.get("stft", {})
        self.stft = CausalSTFT(
            n_fft=stft_cfg.get("n_fft", 256),
            hop_length=stft_cfg.get("hop_size", 80),
        ).to(device)

        self.model.eval()

    @torch.no_grad()
    def process_audio(self, audio_tensor):
        """Process audio tensor and return enhanced version."""
        if audio_tensor.dim() == 1:
            audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0)
        elif audio_tensor.dim() == 2:
            audio_tensor = audio_tensor.unsqueeze(0)

        audio_tensor = audio_tensor.to(self.device)

        # STFT
        noisy_stft = self.stft(audio_tensor.squeeze(0))

        # Enhance
        enhanced_stft, _, _ = self.model(noisy_stft)

        # Inverse STFT
        enhanced_audio = self.stft.inverse(enhanced_stft)

        return enhanced_audio.cpu().squeeze()

    def process_file(self, input_path, output_path=None):
        """Process audio file and optionally save result."""
        audio, sr = torchaudio.load(input_path)
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            audio = torchaudio.transforms.Resample(sr, self.sample_rate)(audio)

        enhanced = self.process_audio(audio)

        if output_path:
            torchaudio.save(output_path, enhanced.unsqueeze(0), self.sample_rate)
            print(f"💾 Saved enhanced audio: {output_path}")

        return audio.squeeze(), enhanced


# Load best model for inference
best_model_path = os.path.join(CONFIG["paths"]["checkpoint_dir"], "best_model.pt")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"✅ Loaded best model from {best_model_path}")

inferencer = AuraNetInference(model, CONFIG, DEVICE)
print("✅ Inference engine ready")

In [ ]:
# =============================================================================
# TEST ON SYNTHETIC NOISY AUDIO
# =============================================================================

# Generate test audio
print("🎵 Generating test audio...")

# Create clean signal (speech-like harmonics)
sample_rate = CONFIG["audio"]["sample_rate"]
duration = 3.0
t = torch.linspace(0, duration, int(sample_rate * duration))

# Simulate speech with harmonics
f0 = 150  # fundamental frequency
clean_audio = torch.zeros_like(t)
for h in range(1, 8):
    clean_audio += (1.0 / h) * torch.sin(2 * 3.14159 * f0 * h * t)

# Add amplitude modulation
clean_audio *= 0.5 + 0.5 * torch.sin(2 * 3.14159 * 3 * t)
clean_audio = clean_audio / clean_audio.abs().max() * 0.5

# Add noise
noise = torch.randn_like(t) * 0.1
noisy_audio = clean_audio + noise

# Process
print("🔄 Processing with AuraNet...")
enhanced_audio = inferencer.process_audio(noisy_audio)

# Display audio players
print("\n🔊 Original Noisy Audio:")
display(Audio(noisy_audio.numpy(), rate=sample_rate))

print("\n🔊 Enhanced Audio:")
display(Audio(enhanced_audio.numpy(), rate=sample_rate))

print("\n🔊 Clean Reference:")
display(Audio(clean_audio.numpy(), rate=sample_rate))

In [ ]:
# =============================================================================
# UPLOAD AND PROCESS YOUR OWN AUDIO
# =============================================================================

print("📤 Upload your audio file (wav, mp3, flac):")
uploaded = files.upload()

if uploaded:
    input_filename = list(uploaded.keys())[0]
    output_filename = f"enhanced_{input_filename}"

    print(f"\n🔄 Processing: {input_filename}")

    # Process
    original, enhanced = inferencer.process_file(input_filename, output_filename)

    # Play original
    print("\n🔊 Original Audio:")
    display(Audio(original.numpy(), rate=CONFIG["audio"]["sample_rate"]))

    # Play enhanced
    print("\n🔊 Enhanced Audio:")
    display(Audio(enhanced.numpy(), rate=CONFIG["audio"]["sample_rate"]))

    # Download enhanced
    print(f"\n⬇️ Downloading enhanced audio...")
    files.download(output_filename)
else:
    print("No file uploaded")

## 🐛 12. Debug Mode Test

In [ ]:
# =============================================================================
# DEBUG MODE - Test with 1 batch
# =============================================================================

def run_single_batch_test():
    """Run a single batch to verify everything works."""
    print("🐛 Running single batch debug test...")

    # Create small test dataset
    test_dataset = AuraNetDataset(
        clean_dirs=[],
        noise_dir=None,
        sample_rate=CONFIG["audio"]["sample_rate"],
        segment_length=1.0,
        synthetic_mode=True,
        num_synthetic=CONFIG["training"]["batch_size"],
    )
    test_loader = DataLoader(test_dataset, batch_size=CONFIG["training"]["batch_size"])

    # Get one batch
    batch = next(iter(test_loader))
    noisy_stft = batch["noisy_stft"].to(DEVICE)
    clean_stft = batch["clean_stft"].to(DEVICE)
    noisy_audio = batch["noisy_audio"].to(DEVICE)
    clean_audio = batch["clean_audio"].to(DEVICE)

    print(f"\n📊 Batch shapes:")
    print(f"  noisy_stft: {noisy_stft.shape}")
    print(f"  clean_stft: {clean_stft.shape}")
    print(f"  noisy_audio: {noisy_audio.shape}")
    print(f"  clean_audio: {clean_audio.shape}")

    # Forward pass
    model.eval()
    with torch.no_grad():
        enhanced_stft, wdrc_params, hidden = model(noisy_stft)

    print(f"\n📤 Output shapes:")
    print(f"  enhanced_stft: {enhanced_stft.shape}")
    print(f"  WDRC attack: {wdrc_params['attack_coeff'].shape}")
    print(f"  hidden: {hidden.shape}")

    # Compute loss
    stft = CausalSTFT().to(DEVICE)
    enhanced_audio = stft.inverse(enhanced_stft)

    criterion = AuraNetLoss()
    loss, loss_dict = criterion(enhanced_stft, clean_stft, enhanced_audio, clean_audio)

    print(f"\n📉 Loss values:")
    for k, v in loss_dict.items():
        print(f"  {k}: {v.item():.4f}")

    print("\n✅ Debug test passed!")
    return True


# Run debug test
run_single_batch_test()

## 💾 13. Download Trained Model

In [ ]:
# =============================================================================
# DOWNLOAD TRAINED MODEL
# =============================================================================

# List available checkpoints
checkpoint_dir = CONFIG["paths"]["checkpoint_dir"]
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    print("📋 Available checkpoints:")
    for cp in sorted(checkpoints):
        path = os.path.join(checkpoint_dir, cp)
        size_mb = os.path.getsize(path) / 1e6
        print(f"  - {cp} ({size_mb:.1f} MB)")

    # Download best model
    best_path = os.path.join(checkpoint_dir, "best_model.pt")
    if os.path.exists(best_path):
        print(f"\n⬇️ Downloading best_model.pt...")
        files.download(best_path)
    else:
        print("\n⚠️ No best_model.pt found")
else:
    print("⚠️ No checkpoints found. Train the model first!")

---

## 📚 Usage Instructions

### Dataset Setup

1. Create folder structure in Google Drive:
```
/My Drive/auranet_dataset/
    /speech/     # Clean speech recordings
    /music/      # Clean music recordings  
    /noise/      # Various noise samples
```

2. Upload audio files (WAV, MP3, FLAC, OGG supported)

3. Run all cells in order

### Debug Mode

Set `CONFIG["debug_mode"] = True` to:
- Use synthetic data (no dataset needed)
- Run only 2 epochs with smaller batches
- Quick validation that everything works

### Resume Training

Training automatically resumes from the last checkpoint if available.

### Inference

1. Upload your noisy audio file
2. Model processes and enhances it
3. Listen to before/after comparison
4. Download enhanced audio

---